In [ ]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental
!pip install -q langchain-openai langchain-core requests

### **Runnable**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch
import random

class NakliLLM:
  def __init__(self):
    print('LLM created')

  def predict(self, prompt):
    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]

    return {'response': random.choice(response_list)}

In [ ]:
class NakliPromptTemplate:
  def __init__(self, template, input_variables):
    self.template = template
    self.input_variables = input_variables

  def format(self, input_dict):
    return self.template.format(**input_dict)

template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

prompt = template.format({'length':'short', 'topic':'india'})
llm = NakliLLM()
llm.predict(prompt)

In [ ]:
class NakliChain:
  def __init__(self, llm, prompt):
    self.llm = llm
    self.prompt = prompt

  def run(self, input_dict):
    final_prompt = self.prompt.format(input_dict)
    result = self.llm.predict(final_prompt)
    return result['response']

template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

llm = NakliLLM()
chain = NakliChain(llm, template)
chain.run({'length': 'five', 'topic': 'dogs'})

In [ ]:
prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()
parser = StrOutputParser()
prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)
print(chain.invoke({'topic':'AI'}))

In [ ]:
prompt1 = PromptTemplate(
    template='Generate a tweet about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a Linkedin post about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()
parser = StrOutputParser()
parallel_chains = RunnableParallel({
    'tweet': RunnableSequence(prompt1, model, parser),
    'linkedin': RunnableSequence(prompt2, model, parser)
})

result = parallel_chains.invoke({'topic':'AI'})
print(result['tweet'])
print(result['linkedin'])

In [ ]:
prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()
parser = StrOutputParser()
prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

joke_gen_chain = RunnableSequence(prompt1, model, parser)
parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(prompt2, model, parser)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)
print(final_chain.invoke({'topic':'cricket'}))

In [ ]:
def word_count(text):
  return len(text.split())

prompt = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()
parser = StrOutputParser()
joke_gen_chain = RunnableSequence(prompt, model, parser)
parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_count': RunnableLambda(word_count)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)
result = final_chain.invoke({'topic':'AI'})
final_result = """{} \n word count - {}""".format(result['joke'], result['word_count'])
print(final_result)

In [ ]:
prompt1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)
prompt2 = PromptTemplate(
    template='Summarize the following text \n {text}',
    input_variables=['text']
)

model = ChatOpenAI()
parser = StrOutputParser()
report_gen_chain = prompt1 | model | parser
branch_chain = RunnableBranch(
    (lambda x: len(x.split())>300, prompt2 | model | parser),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)
print(final_chain.invoke({'topic':'Russia vs Ukraine'}))

### **Tools: Toolskit and Tool Calling**

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import ShellTool
from langchain_core.tools import tool
from langchain.tools import StructuredTool, BaseTool
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from typing import Type

search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('top news in Bangladesh today')
print(results)
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

In [ ]:
shell_tool = ShellTool()
results = shell_tool.invoke('ls')
print(results)

In [ ]:
def multiply(a, b):
  return a*b

def multiply(a: int, b: int) -> int:
  return a*b

@tool
def multiply(a: int, b: int) -> int:
  return a*b

result = multiply.invoke({"a":3, "b":5})
print(result)
print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

In [ ]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description='first number to add')
  b: int = Field(required=True, description='second number to add')

def multiply_func(a: int, b: int) -> int:
  return a*b

mutiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name='multiply',
    description='multiply two numbers',
    args_schema=MultiplyInput
)

result = mutiply_tool.invoke({'a':3, 'b':5})
print(result)
print(mutiply_tool.name)
print(mutiply_tool.description)
print(mutiply_tool.args)

In [ ]:
class MultiplyTool(BaseTool):
  name: str = "multiply"
  description: str = "multiply two numbers"
  args_schema: Type[BaseModel] = MultiplyInput
  def _run(self, a: int, b: int) -> int:
    return a*b

multiply_tool = MultiplyTool()
result = multiply_tool.invoke({'a':3, 'b':5})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

In [ ]:
@tool
def add(a: int, b: int) -> int:
  return a+b

@tool
def multiply(a: int, b: int) -> int:
  return a*b

class MathToolkit:
  def get_tools(self):
    return [add, multiply]

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
  print(tool.name, "=>", tool.description)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

@tool
def multiply(a: int, b: int) -> int:
  return a*b

print(multiply.invoke({'a':3, 'b':4}))
print(multiply.name)
print(multiply.description)
print(multiply.args)

In [ ]:
llm = ChatOpenAI()
llm.invoke('hi')
llm_with_tools = llm.bind_tools({multiply})
llm_with_tools.invoke('Hi how are you')
query = HumanMessage('can you multiply 3 with 1000')
messages = [query]
messages

In [ ]:
result = llm_with_tools.invoke(messages)
messages.append(result)
tool_result = multiply.invoke(result.tool_calls[0])
messages.append(tool_result)
tool_result, messages

In [ ]:
from typing import Annotated
from langchain_core.tools import InjectedToolArg

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  return base_currency_value * conversion_rate

convert.args

In [ ]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

In [ ]:
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

messages

In [ ]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)
ai_message.tool_calls

In [ ]:
import json

for tool_call in ai_message.tool_calls:
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)

  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

messages, llm_with_tools.invoke(messages).content

In [ ]:
from langchain.agents import initialize_agent, AgentType

agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

user_query = "Hi how are you?"
response = agent_executor.invoke({"input": user_query})
response

### **LangSmith Tutorial: Setup**

In [ ]:
prompt = PromptTemplate.from_template("{question}")
model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt | model | parser
result = chain.invoke({"question": "What is the capital of Peru?"})
print(result)

In [ ]:
prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)
prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt1 | model | parser | prompt2 | model | parser
result = chain.invoke({'topic': 'Unemployment in Bangladesh'})
print(result)